# Neural Simulation Pipeline: Looming Response Calculation
This script executes a high-throughput simulation to calculate neuronal population responses to Looming (Expanding Disk) stimuli using the Signal Infinite Cascade (SIC) model.

**Core Functionalities:**

* **Stimulus Standardizing:** Processes raw visual inputs into a standardized $41 \times 82$ spatio-temporal grid at $1.0\,ms$ resolution.
* **Circuit-Level Simulation:** Loads connectome-derived synaptic weights for a specific population of neurons (defined by `root_id`).
* **Steady-State Integration:** Implements a 50-step baseline stabilization phase before stimulus onset to ensure accurate dynamic modeling.

In [ ]:
import sys
import os

# -----------------------------
# DEPENDENCIES
# -----------------------------
sys.path.append(os.path.abspath('../util'))

from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import csv
import glob
import torch

# ==================================================
# Result directory (npz)
# ==================================================
RESPONSE_ROOT = "../results/responses/loom"

# ==================================================
# Global npz deduplication (scan ALL response folders)
# ==================================================
print("Scanning existing npz files for global deduplication...")
existing_npz = set()

for npz_file in glob.glob(
    os.path.join(RESPONSE_ROOT, "**", "*.npz"),
    recursive=True
):
    base = os.path.splitext(os.path.basename(npz_file))[0]
    existing_npz.add(base)

print(f"Found {len(existing_npz)} existing response files")

# ---------------------------
# Initialize StimulusProcessor
# ---------------------------
print("\nInitializing StimulusProcessor...")
stimulus_processor = StimulusProcessor(
    target_dt_ms=1.0,
    target_size=(41, 82),
    is_visual=False,
    fps=1000.0
)

# ---------------------------
# Load stimulus videos (AFTER global npz dedup)
# ---------------------------
video_files = sorted(glob.glob("../stimulus/LoomingDarkDisk/*/npz/*.npz"))

if not video_files:
    raise RuntimeError("❌ No .npz files found.")

print(f"Found {len(video_files)} video files")

stimulus_dataset = {}
skipped = 0

print("\nLoading stimuli (global npz dedup)...")
for v_file in tqdm(video_files, desc="Loading stimuli", unit="video"):

    stim_base = os.path.splitext(os.path.basename(v_file))[0]

    tau_ms = os.path.normpath(v_file).split(os.sep)[-3]  
    stim_name_with_tau = f"{stim_base}_{tau_ms}"

    if stim_name_with_tau in existing_npz:
        skipped += 1
        continue

    stimulus, _ = stimulus_processor.process_npz(
        v_file,
        verbose=False
    )

    stimulus_dataset[stim_name_with_tau] = stimulus

print(f"✅ Stimuli loaded: {len(stimulus_dataset)}")
print(f"⏭️  Stimuli skipped (existing npz anywhere): {skipped}")

# ---------------------------
# Initialize LMCs model (AFTER stimulus filtering)
# ---------------------------
print("\nInitializing LMCs model...")
SIC_model = SICModelTorch(
    matrix_dir="neuron_matrices",
    output_dir="../results/responses/fri",
    t_step=10,
    rate=100,
    device=torch.device("cuda:0")
)
# ---------------------------
# Load neuron IDs
# ---------------------------
def read_all_neuron_ids(filename='../data/classification.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")
    return neuron_ids

print("\nLoading neuron weights...")

neuron_ids = [720575940615659521,720575940620918789,720575940608188427,720575940646764579,720575940644659236,720575940629438508,720575940631609391,720575940612014131,720575940632043576,720575940632903737,720575940629413962,720575940630839371,720575940617904208,720575940637966424,720575940636606554,720575940618870881,720575940610236514,720575940622540899,720575940630290536,720575940630610042,720575940628398202,720575940621181066,720575940623556755,720575940622925972,720575940618666142,720575940655587489,720575940639170723,720575940620714150,720575940614660263,720575940628881576,720575940632412333,720575940625006776,720575940615495867,720575940634599615,720575940609589443,720575940621304004,720575940634140883,720575940622565590,720575940618051801,720575940642545883,720575940638204132,720575940613636325,720575940625596647,720575940641480949,720575940653310198,720575940628971779,720575940625047815,720575940621336852,720575940627087646,720575940622147873,720575940630135084,720575940620632369,720575940617290043,720575940627210568,720575940622500172,720575940626399588,720575940629725545,720575940649992569,720575940627997050,720575940622352764,720575940660224385,720575940618936710,720575940622123400,720575940623786376,720575940614840715,720575940615692690,720575940614185366,720575940628627867,720575940630274459,720575940614562207,720575940611498419,720575940617568697,720575940615578043,720575940605714878,720575940609376707,720575940630569413,720575940631110095,720575940620804566,720575940617249254,720575940620468718,720575940629586417,720575940635640312,720575940627595772,720575940627538434,720575940616471052,720575940631282194,720575940644438551,720575940610605614,720575940631855663,720575940616167986,720575940638810677,720575940619944513,720575940630078031,720575940639425112,720575940626268761,720575940636254810,720575940635222640,720575940621804159,720575940639539840,720575940646814340,720575940621107847,720575940634583694,720575940623934096,720575940624319124,720575940634428058,720575940602864300,720575940617552569,720575940614046395,720575940616274619,720575940627055303,720575940628800207,720575940608877266,720575940612629210,720575940622910188,720575940614759149,720575940627620593,720575940626187014,720575940609417995,720575940627366675,720575940631593753,720575940619928346,720575940622263066,720575940644758308,720575940624982823,720575940629390123,720575940625376062,720575940634526560,720575940633543531,720575940634207087,720575940613358450,720575940621599604,720575940649829241,720575940621566844,720575940659356545,720575940626088835,720575940647478148,720575940641833869,720575940634198926,720575940625539984,720575940625744784,720575940621304723,720575940623491987,720575940609844117,720575940624106388,720575940618519454,720575940620231590,720575940611654570,720575940605658033,720575940624655285,720575940619133878,720575940618863553,720575940620878785,720575940645176264,720575940612203482,720575940632339425,720575940633781217,720575940628030440,720575940620534768,720575940641276917,720575940636656632,720575940623467527,720575940630758418,720575940623918121,720575940629472298,720575940632355885,720575940620543025,720575940631520312,720575940633011256,720575940631168057,720575940632486969,720575940621534273,720575940622550089,720575940627645514,720575940620149836,720575940608402524,720575940625187940,720575940628669545,720575940630119532,720575940629742718,720575940618314897,720575940622632084,720575940633953434,720575940618658974,720575940632634532,720575940631987364,720575940613088426,720575940613915823,720575940621640885,720575940614235323,720575940623033532,720575940607632579,720575940607378633,720575940625482954,720575940611531986,720575940632626402,720575940627432680,720575940622836970,720575940629521655,720575940627825923,720575940627924227,720575940628825348,720575940624991495,720575940626834698,720575940621157645,720575940623164685,720575940625343761,720575940625253649,720575940626859283,720575940620354836,720575940627285267,720575940645045527,720575940618167579,720575940606633248,720575940626982181,720575940623385894,720575940623426854,720575940615251251,720575940647560500,720575940630291767,720575940626580794,720575940629792060,720575940629833020,720575940639802685,720575940613170498,720575940632307025,720575940634862943,720575940634903915,720575940645258606,720575940640589171,720575940627850622,720575940621313414,720575940631061900,720575940631643532,720575940613793170,720575940614342038,720575940641793440,720575940638623165,720575940625573322,720575940614186449,720575940606756309,720575940620404182,720575940622321110,720575940630480348,720575940622067165,720575940631651810,720575940622042603,720575940623992300,720575940611474925,720575940630382075,720575940615505406,720575940627482114,720575940627015196,720575940626179614,720575940627629598,720575940624229925,720575940617750070,720575940631283267,720575940617373268,720575940614587990,720575940620297819,720575940637648485,720575940615898730,720575940636329579,720575940622386824,720575940631225996,720575940623582867,720575940627490459,720575940632512156,720575940619191966,720575940638664355,720575940613686954,720575940619486891,720575940604470956,720575940604954289,720575940625065653,720575940624426680,720575940616185531,720575940635838143,720575940620322497,720575940611237594,720575940637607646,720575940632962786,720575940618594022,720575940621002480,720575940628498161,720575940626278129,720575940653614838,720575940624402173,720575940622681864,720575940625090314,720575940631299855,720575940628858640,720575940625573653,720575940631627551,720575940624967463,720575940613408552,720575940615030568,720575940629866283,720575940624877360,720575940617873201,720575940618405686,720575940619757366,720575940640630589,720575940618594117,720575940624131913,720575940632422219,720575940630054735,720575940632037224,720575940635166571,720575940630505334,720575940626319235,720575940617922439,720575940625467280,720575940624861072,720575940611622826,720575940633995181,720575940614014895,720575940612335539,720575940623222712,720575940622673860,720575940622944196,720575940619945931,720575940623132621,720575940630980559,720575940614129617,720575940620625880,720575940615292895,720575940625295339,720575940624082931,720575940639705077,720575940614187006,720575940622878719,720575940622010373,720575940630808591,720575940627949589,720575940645693476,720575940625426469,720575940614236200,720575940631889965,720575940614137907,720575940629686339,720575940627826760,720575940629170250,720575940628252751,720575940628883535,720575940629956695,720575940626229337,720575940634617952,720575940605282406,720575940631545960,720575940645537902,720575940610173048,720575940619077755,720575940624459907,720575940618086535,720575940626901136,720575940623468691,720575940614736034,720575940636592302,720575940613589167,720575940621068464,720575940619372726,720575940607854793,720575940611016910,720575940608960722,720575940615219417,720575940612376805,720575940630808827,720575940624394498,720575940613998860,720575940644022551,720575940621519130,720575940630071583,720575940620380449,720575940647471395,720575940644751652,720575940623706405,720575940644653348,720575940647029044,720575940630956345,720575940618291520,720575940610001220,720575940631333195,720575940624968013,720575940626729295,720575940637501784,720575940635175263,720575940623968612,720575940614588778,720575940613007729,720575940620110215,720575940622838154,720575940616374667,720575940634560922,720575940634864026,720575940626639259,720575940643817888,720575940604103084,720575940637870510,720575940604930481,720575940626672058,720575940622797244,720575940624484796,720575940620929483,720575940622027222,720575940630333911,720575940603980256,720575940615899622,720575940642449909,720575940629064186,720575940626557442,720575940624280066,720575940622248453,720575940627417619,720575940620298772,720575940626524702,720575940604783136,720575940644801060,720575940617390641,720575940646160948,720575940626663994,720575940617792069,720575940626295396,720575940637608560,720575940610935410,720575940629777018,720575940621257340,720575940620642940,720575940619790975,720575940641237632,720575940626745987,720575940626377349,720575940640156301,720575940617554577,720575940634782359,720575940635355804,720575940621388453,720575940636879534,720575940613917359,720575940619315888,720575940620470976,720575940622805700,720575940639353552,720575940630244060,720575940622633716,720575940620757748,720575940633201400,720575940628441862,720575940627368723,720575940633488153,720575940623338281,720575940629490476,720575940631759663,720575940618398533,720575940623543117,720575940635872101,720575940614572906,720575940619905916,720575940617448331,720575940629965708,720575940614843275,720575940644473760,720575940615023535,720575940610091955,720575940612311987,720575940619676598,720575940617030585,720575940608076739,720575940615424966,720575940627139530,720575940617399257,720575940633095137,720575940625345511,720575940625673196,720575940620749807,720575940627827697,720575940641278965,720575940631342073,720575940628745209,720575940614089726,720575940617432065,720575940625353736,720575940622478349,720575940628884496,720575940621626394,720575940632161311,720575940625345584,720575940638084149,720575940625550398,720575940621339713,720575940622486644,720575940614384757,720575940650929273,720575940623125628,720575940626328707,720575940647586948,720575940625878160,720575940616604817,720575940615957663,720575940614581410,720575940612557994,720575940612648106,720575940605725873,720575940635552948,720575940618382529,720575940630334658,720575940626877639,720575940625558730,720575940633529548,720575940611886289,720575940606856405,720575940610231509,720575940620135645,720575940604284128,720575940622806247,720575940623355114,720575940641148149,720575940627500279,720575940636454136,720575940626238716,720575940623781127,720575940629736720,720575940632472850,720575940627303699,720575940620823828,720575940617809181,720575940622437671,720575940622929193,720575940608838958,720575940615679282,720575940618440000,720575940615671106,720575940619332944,720575940631973207,720575940626189657,720575940637101413,720575940631629160,720575940621667693,720575940618947956,720575940637126007,720575940633218444,720575940614352278,720575940635405719,720575940610010521,720575940629155227,720575940615630237,720575940615794077,720575940611976618,720575940605300140,720575940612058543,720575940620561840,720575940621471157,720575940640140733,720575940619152856,720575940622675421,720575940616293855,720575940603874784,720575940630425083,720575940622347784,720575940616826380,720575940628360723,720575940626296339,720575940645047831,720575940623019544,720575940620815896,720575940623142426,720575940603489824,720575940645834275,720575940631309866,720575940613590579,720575940616498742,720575940614909506,720575940639428174,720575940633996883,720575940621119061,720575940616400470,720575940632989291,720575940638019184,720575940614147701,720575940647964281,720575940618514043,720575940625182341,720575940612427414,720575940619038366,720575940628885160,720575940623068841,720575940611772083,720575940621700789,720575940615958203,720575940630990533,720575940627099338,720575940639919824,720575940610715346,720575940617031385,720575940642549467,720575940621274845,720575940623781611,720575940626165489,720575940650282742,720575940628942586,720575940626009852,720575940627582724,720575940632407826,720575940619038493,720575940629327659,720575940625870640,720575940630236988,720575940631498563,720575940611108676,720575940627828552,720575940627812168,720575940628352850,720575940619431765,720575940616597334,720575940640747352,720575940627787609,720575940621561697,720575940625747812,720575940636651376,720575940630810486,720575940630695798,720575940626730883,720575940648636292,720575940624748419,720575940627107721,720575940617498507,720575940627771279,720575940612427670,720575940634750871,720575940619341744,720575940605513649,720575940624977847,720575940623052728,720575940625641402,720575940632620991,720575940642312136,720575940619530187,720575940623503318,720575940637618142,720575940616286175,720575940604727264,720575940616376287,720575940638109668,720575940625887207,720575940623110123,720575940627722234,720575940612804606,720575940607971337,720575940608716809,720575940626665489,720575940620300308,720575940621586473,720575940631662648,720575940616548416,720575940617990228,720575940621021278,720575940642926702,720575940636610671,720575940645556334,720575940637667443,720575940628467830,720575940641173632,720575940660269185,720575940659703937,720575940645032068,720575940622594186,720575940613632153,720575940626919579,720575940620054694,720575940628844712,720575940630712493,720575940631498925,720575940612698287,720575940624986300,720575940637896893,720575940607037635,720575940630106309,720575940642369736,720575940632178892,720575940610863310,720575940628582607,720575940612206810,720575940633293025,720575940610199781,720575940613558509,720575940618440943,720575940612804862,720575940625314058,720575940626092302,720575940631122191,720575940627001637,720575940627542316,720575940639314229,720575940630868281,720575940619432261,720575940638175558,720575940633727339,720575940640723315,720575940630671734,720575940636479863,720575940660220289,720575940620169607,720575940624404872,720575940615623051,720575940633178508,720575940635701655,720575940614107554,720575940639789475,720575940631925156,720575940612288947,720575940619751872,720575940621554112,720575940625502666,720575940617802187,720575940611813842,720575940631695841,720575940632646114,720575940631720417,720575940622332403,720575940621226484,720575940635587064,720575940612805118,720575940629139968,720575940616786433,720575940626936324,720575940628869648,720575940620915224,720575940625396252,720575940627419676,720575940626993694,720575940603580960,720575940614959675,720575940616401467,720575940618203707,720575940632302151,720575940627845704,720575940621783628,720575940637659749,720575940636308075,720575940638159472,720575940617663093,720575940620038779,720575940645032580,720575940621767306,720575940632539788,720575940641305229,720575940635464343,720575940616688286,720575940619678366,720575940644467360,720575940639789731,720575940621071017,720575940620595883,720575940622160560,720575940611756723,720575940625420986,720575940630221509,720575940615385798,720575940610872014,720575940614165201,720575940614181585,720575940633170643,720575940629041879,720575940612682467,720575940627731176,720575940621325034,720575940623626986,720575940639159029,720575940630221563,720575940621947653,720575940622537477,720575940606784265,720575940628345610,720575940631270162,720575940605702944,720575940642894628,720575940624986919,720575940615058216,720575940629369642,720575940629312300,720575940627403578,720575940616647483,720575940640142141,720575940625642302,720575940633228103,720575940628378440,720575940626518858,720575940623446860,720575940626527055,720575940618818395,720575940631982952,720575940634915691,720575940635210603,720575940636029807,720575940639634304,720575940608332674,720575940628132745,720575940623528842,720575940633228172,720575940640306061,720575940619654053,720575940620153765,720575940622037926,720575940615607207,720575940622537653,720575940624651189,720575940625937340,720575940613731270,720575940606866377,720575940606342089,720575940617802699,720575940609733582,720575940611765201,720575940617245652,720575940617917396,720575940623274966,720575940636898270,720575940614075363,720575940625306603,720575940621530117,720575940606702601,720575940618548244,720575940644279319,720575940632409130,720575940625863728,720575940647031860,720575940647228468,720575940631499832,720575940632900665,720575940640486488,720575940626068569,720575940622996579,720575940614403178,720575940635210859,720575940619089005,720575940611552370,720575940630697078,720575940629812346,720575940640978048,720575940646753412,720575940621399174,720575940621071495,720575940641518733,720575940627731599,720575940634825879,720575940626904219,720575940614730914,720575940621628582,720575940606112940,720575940620473547,720575940623193293,720575940609316050,720575940632868051,720575940642198747,720575940637603038,720575940631090402,720575940626027752,720575940619744496,720575940613403890,720575940628731140,720575940623938823,720575940630803727,720575940632458514,720575940631549202,720575940623037720,720575940626748702,720575940626806046,720575940631483695,720575940628436282,720575940631721287,720575940623537485,720575940628796780,720575940645352814,720575940649809273,720575940629304702,720575940639479168,720575940660229505,720575940620621190,720575940641781133,720575940642411917,720575940620711334,720575940610348458,720575940605162924,720575940620776880,720575940603786686,720575940613838289,720575940622161366,720575940617696728,720575940617295321,720575940615378399,720575940624897516,720575940621817326,720575940623062515,720575940639159797,720575940630033913,720575940621923837,720575940622505511,720575940631705130,720575940630492714,720575940632499757,720575940631959097,720575940626888250,720575940637660733,720575940638094911,720575940617918021,720575940639118926,720575940621031006,720575940612019825,720575940612028017,720575940620760692,720575940631508598,720575940608145026,720575940627166851,720575940607170178,720575940648654468,720575940621121159,720575940628264585,720575940636333710,720575940611225237,720575940610610841,720575940617950877,720575940653053601,720575940614682274,720575940625094325,720575940623431351,720575940626085562,720575940637308605,720575940618147520,720575940624537284,720575940623128269,720575940640331472,720575940617074393,720575940611462885,720575940611102437,720575940625733351,720575940626732776,720575940624766700,720575940651341558,720575940624234242,720575940615378700,720575940632524559,720575940632041247,720575940611700520,720575940627003179,720575940627560236,720575940625872688,720575940615092019,720575940631082808,720575940629509948,720575940613535554,720575940631271235,720575940628993871,720575940628387663,720575940632303441,720575940638013274,720575940636186463,720575940630689644,720575940630607738,720575940618393478,720575940627732361,720575940616353675,720575940624750480,720575940612429714,720575940623431571,720575940613019541,720575940618876830,720575940618663838,720575940654094241,720575940630992813,720575940612429743,720575940632573887,720575940627691458,720575940627732423,720575940642027464,720575940606425033,720575940613748689,720575940607178709,720575940630403031,720575940604729312,720575940634253281,720575940613347299,720575940637530084,720575940637341668,720575940621801456,720575940628445169,720575940624816115,720575940629428215,720575940630665210,720575940608735241,720575940625815562,720575940625184782,720575940628125715,720575940616886301,720575940643837988,720575940628895787,720575940611610670,720575940609488942,720575940639242303,720575940636104767,720575940632991815,720575940630059087,720575940605450342,720575940615108714,720575940631124076,720575940627814522,720575940619548799,720575940640422016,720575940618827910,720575940623276168,720575940621629578,720575940631632012,720575940615043222,720575940617296029,720575940619024542,720575940639201443,720575940604737708,720575940611881139,720575940615715003,720575940620564673,720575940633213132,720575940631091407,720575940610472146,720575940621392086,720575940611782874,720575940602534112,720575940612036837,720575940639152361,720575940620859630,720575940625144061,720575940622579967,720575940625422609,720575940627167516,720575940627921182,720575940615043368,720575940628437291,720575940629502251,720575940629477675,720575940610505006,720575940646443316,720575940629412170,720575940622760269,720575940632934751,720575940603591014,720575940606179686,720575940627241321,720575940635212139,720575940618762605,720575940617140597,720575940639553920,720575940616935838,720575940644542880,720575940644166048,720575940625963432,720575940613929391,720575940619024816,720575940623137212,720575940629461442,720575940629240263,720575940623899082,720575940634163682,720575940614363619,720575940624882151,720575940623776235,720575940629543409,720575940651932150,720575940630829563,720575940629486075,720575940615453182,720575940627044885,720575940605950496,720575940611807795,720575940639726141,720575940624374334,720575940632836675,720575940631829059,720575940627847752,720575940619434587,720575940633041503,720575940634458719,720575940619090541,720575940609366648,720575940623153788,720575940648188548,720575940621294218,720575940611308181,720575940613307033,720575940630428324,720575940612586151,720575940605041330,720575940607384242,720575940613814971,720575940623203004,720575940629846722,720575940630297292,720575940610341582,720575940615559892,720575940619590360,720575940617689816,720575940619582168,720575940613339875,720575940623637226,720575940619524852,720575940621196037,720575940615502604,720575940626889491,720575940618427156,720575940627946261,720575940621572890,720575940618156829,720575940617952034,720575940616936226,720575940625718053,720575940623014695,720575940624349993,720575940632877869,720575940645534516,720575940631051065,720575940616026939,720575940617296699,720575940613200706,720575940614478658,720575940617034565,720575940640316238,720575940607761244,720575940635179871,720575940635564896,720575940609948514,720575940628659049,720575940620622708,720575940629805950,720575940618402694,720575940625513360,720575940612692889,720575940618574749,720575940617477022,720575940637424547,720575940622195625,720575940610309034,720575940637612974,720575940612373423,720575940606704562,720575940619910080,720575940609407939,720575940632189900,720575940609260494,720575940631108559,720575940609719250,720575940611939290,720575940615666655,720575940632574946,720575940624530407,720575940624333802,720575940628487153,720575940621319156,720575940652104694,720575940630682618,720575940619877396,720575940610604078,720575940646173748,720575940632788025,720575940637678653,720575940626816062,720575940610677828,720575940632296519,720575940628397138,720575940621065301,720575940606753884,720575940629421159,720575940627307625,720575940628659305,720575940630346860,720575940629757036,720575940634229871,720575940611644529,720575940616649845,720575940648180868,720575940621401222,720575940627790991,720575940628225167,720575940617280657,720575940613717138,720575940612488341,720575940614642838,720575940634459290,720575940643347616,720575940644740256,720575940620541099,720575940605009097,720575940610440402,720575940628241628,720575940625489160,720575940619074836,720575940627234069,720575940622204186,720575940632632607,720575940644666660,720575940626799920,720575940638088511,720575940627750216,720575940624121161,720575940620082512,720575940632722771,720575940618870113,720575940622040419,720575940620582260,720575940631518582,720575940620156283,720575940617993607,720575940624407944,720575940621270410,720575940631469452,720575940641832333,720575940630289804,720575940617035153,720575940612603289,720575940644494752,720575940611087786,720575940618550699,720575940624448956,720575940638637501,720575940605656521,720575940611735002,720575940612922851,720575940611956197,720575940624874983,720575940625890812,720575940621082111,720575940622728728,720575940619116061,720575940627021349,720575940629782072,720575940638989894,720575940625890888,720575940619451989,720575940639088216,720575940636933722,720575940628577897,720575940620017261,720575940643487342,720575940620369517,720575940621246068,720575940616076917,720575940631543414,720575940634279543,720575940634525303,720575940633853559,720575940622351996,720575940623498876,720575940622843530,720575940633165452,720575940614348438,720575940615323298,720575940618698411,720575940618706603,720575940632698541,720575940617035449,720575940606222014,720575940632379071,720575940615798470,720575940623122125,720575940640128720,720575940631035603,720575940609097429,720575940633771745,720575940611981029,720575940622917351,720575940621909739,720575940627799802,720575940613816062,720575940629012227,720575940622933765,720575940627652358,720575940626849546,720575940608409355,720575940613668620,720575940622786317,720575940626562833,720575940622237466,720575940632960813,720575940640325429,720575940632043320,720575940636098367,720575940636761926,720575940618182480,720575940627414887,720575940647755641,720575940620328828,720575940627496829,720575940629806974,720575940606607234,720575940636671886,720575940656144289,720575940635549620,720575940623925173,720575940624015287,720575940625047479,720575940640194493,720575940629774274,720575940607451081,720575940630085583,720575940615364564,720575940611735514,720575940624801771,720575940624154604,720575940612366322]
print(f"Found {len(neuron_ids)} neurons")

weights = SIC_model.load_weights(neuron_ids, normalize=False)
print("Weights loaded successfully")

# ---------------------------
# Calculate responses
# ---------------------------
print("\nCalculating LMC responses...")
for stim_name, stimulus in tqdm(
    stimulus_dataset.items(),
    desc="Processing responses",
    unit="stimulus"
):
    SIC_model.calculate_response_baseline(
        stimulus,
        weights,
        responce_threshold=0,
        stim_name=stim_name,
        baseline_steps=50
    )

# ---------------------------
# Summary
# ---------------------------
print("\n" + "=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print(f"Total video files found: {len(video_files)}")
print(f"Stimuli skipped (existing npz): {skipped}")
print(f"Stimuli processed: {len(stimulus_dataset)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)
